# Transformer 
注意力机制：   
1.首先将输入X经过embedding层使得X具有坐标信息   
2.然后对X进行多头注意力机制，将X进行多头注意力机制（一层线性投影），得到$Q_i$（查询需要关注的词），$K_i$（被查询的特征），$V_i$（真正的语义）     
3.$S_i = Q_i*K_i^T/sqrt(d_k)$   
4.掩码（可选）：对 padding 或未来位置添加极大负数（因果/填充屏蔽）。    
5.进行softmax，$P_i=softmax(S_i)$   
6.$Y_i=P_i*V_i$（最终包含上下文信息的语义）   
7.将$Y_i$拼接得到Y再进行一次线性映射


# Self-Supervised Learning
首先将原图随机变换为两图，再经过f编码器提取图像特征，再通过g生成向量z，最终要使得两个z尽可能相似。  
![](./images/simclr_fig2.png)
在训练完成后，丢弃g，得到能够提取图像特征的f编码器。   
    
原本不太理解为什么要加个g然后训练完成后丢掉，个人想法：f的目标是提取图像特征，变化后的两图在特征上应该是要有不同的，但是两图的本质是相同的。而g和loss相当于判断器来判断两个特征是否指向相同本质。因此在训练完成之后这个判断器g应被舍弃，留下能提取特征的f

#  DDPM

前向过程就是不断的添加噪声最终，使得图片完全变成噪声。（噪声满足$ N(0,1) $）  
逆向过程则是不断从噪声恢复图片。  
而其中U‑Net模型的作用则是输入$x_t$得到$\epsilon_\theta$以此将$x_{t-1}$还原出来
### 正向过程
$$ 
x_t = \sqrt{\beta_t} x_{t-1} + \sqrt{1 - \beta_t} \epsilon_\theta
$$
以此得到$x_t$，然后将其输入到U-Net模型中，得到$\epsilon_\theta$。使用下面的loss模型进行训练。   
$$
L_{\text{simple}}(\theta) := \mathbb{E}_{t, x_0, \epsilon} \left[ \left\| \epsilon - \epsilon_\theta \left( \sqrt{\bar{\alpha}_t} x_0 + \sqrt{1 - \bar{\alpha}_t} \, \epsilon, \, t \right) \right\|^2 \right]
$$


### 逆向过程

$$
q(x_{t-1} \mid x_t, x_0) = \mathcal{N}\left( x_{t-1}; \, \tilde{\mu}_t(x_t, x_0), \, \tilde{\beta}_t I \right)
$$
其中：
$$
\tilde{\mu}_t(x_t, x_0) := \frac{\sqrt{\bar{\alpha}_{t-1}} \beta_t}{1 - \bar{\alpha}_t} x_0 + \frac{\sqrt{\alpha_t} (1 - \bar{\alpha}_{t-1})}{1 - \bar{\alpha}_t} x_t
$$
并且：
$$
\tilde{\beta}_t := \frac{1 - \bar{\alpha}_{t-1}}{1 - \bar{\alpha}_t} \beta_t
$$
上述公式阐述如何用$x_0$和$x_t$得到$x_{t-1}$   
而$x_0$可以通过由U-net输出的$\epsilon_\theta$得到
然后不断重复这个过程最终得到$x_0$   

# clip
通过对海量图文对进行对比学习，学习一个共享的多模态语义空间，使图像与文本在该空间中可直接比相似度。  
![](clip.png)  
CLIP 采用双编码器结构,图像分支通常为 ViT 或改造的 ResNet,文本分支使用Transformer 语言编码器。将两个的输出使用一下loss模型进行训练
$$
\mathcal{L}_{\text{i2t}} = -\frac{1}{N} \sum_{i=1}^{N} \log \frac{\exp(S_{ii})}{\sum_{j=1}^{N} \exp(S_{ij})}
$$

$$
\mathcal{L}_{\text{t2i}} = -\frac{1}{N} \sum_{j=1}^{N} \log \frac{\exp(S_{jj})}{\sum_{i=1}^{N} \exp(S_{ij})}
$$

$$
\mathcal{L} = \frac{1}{2} \left( \mathcal{L}_{\text{i2t}} + \mathcal{L}_{\text{t2i}} \right)
$$
（相似度S采用缩放余弦形式）


# Dino
DINO 采用学生—教师双网络框架，学生与教师具有相同的骨干模型与投影头。
### loss计算
$T_s, T_t$ 为学生与教师网络温度参数。较小的 $T_t$ 使教师分布更尖锐以提供明确目标,较大的 $T_s$ 能稳定学生训练
$$
p_s = \text{softmax}\left(\frac{z_s}{T_s}\right), \quad
p_t = \text{softmax}\left(\frac{z_t - c}{T_t}\right)
$$
$$
\mathcal{L} = \mathrm{H}(p_t, p_s) = -\sum_i p_t(i) \log p_s(i)
$$
### 参数更新
教师参数更新方法
$$
\theta_t \leftarrow \lambda \, \theta_t + (1 - \lambda) \, \theta_s
$$
避免模型塌缩，同时教师更新慢于学生。

